In [ ]:
import pandas as pd
import json
import seaborn as sns
import matplotlib.pyplot as plt
import transformers 
import glob
import numpy as np 
from functools import reduce

In [ ]:
def load_dataset(clean=True):

    
    with open("../data/train_data.json", "r") as f:
        train = json.load(f)
    with open("../data/val_data.json", "r") as f:
        val = json.load(f)
    with open("../data/test_data.json", "r") as f:
        test = json.load(f)
    
    train_df = pd.DataFrame(train["data"])
    val_df = pd.DataFrame(val["data"])
    test_df = pd.DataFrame(test["data"])
    
    if clean:
        train_len_before = len(train_df)
        val_len_before = len(val_df)
        test_len_before = len(test_df)
        
        train_df = train_df.dropna(subset=['text'])
        val_df = val_df.dropna(subset=['text'])
        test_df = test_df.dropna(subset=['text'])
        
        train_removed = train_len_before - len(train_df)
        val_removed = val_len_before - len(val_df)
        test_removed = test_len_before - len(test_df)
        
        print(f"Removed {train_removed} rows with NaN values from training dataset")
        print(f"Removed {val_removed} rows with NaN values from validation dataset")
        print(f"Removed {test_removed} rows with NaN values from testing dataset")
        print(f"Total removed: {train_removed + val_removed + test_removed} rows")
    
    return {'train': train_df, 'val': val_df, 'test': test_df}

In [ ]:
dataset= load_dataset(clean=True)

In [ ]:
test_df

In [ ]:
test_df = dataset["test"]

In [ ]:
prediction_files = glob.glob("../results/*_predictions.csv")

In [ ]:
prediction_files

In [ ]:
bert_pred = pd.read_csv('../results/bert_predictions.csv')
roberta_pred = pd.read_csv('../results/roberta_predictions.csv')
deberta_pred = pd.read_csv('../results/deberta_predictions.csv')
mini_trasf_pred = pd.read_csv('../results/all-MiniLM-L6-v2_predictions.csv')
gemma2_pred = pd.read_csv('../results/gemma2_predictions.csv')
gemma3_pred = pd.read_csv('../results/gemma3_predictions.csv')

In [ ]:
deberta_pred

In [ ]:
deberta_pred["high_risk_to_neutral"] = (
    (deberta_pred["true_label"] == 2) & (deberta_pred["predicted_label_deberta"] == 0)
)

In [ ]:
deberta_pred["neutral_to_high_risk"] = (
    (deberta_pred["true_label"] == 0) & (deberta_pred["predicted_label_deberta"] == 2)
)

In [ ]:
high_risk_to_neutral_df = deberta_pred[deberta_pred["high_risk_to_neutral"] == True]
print(high_risk_to_neutral_df)

In [ ]:
neutral_to_high_risk_df = deberta_pred[deberta_pred["neutral_to_high_risk"] == True]
print(neutral_to_high_risk_df)

In [ ]:
deberta_pred

In [ ]:
import os
dfs = []

for file in prediction_files:
    model_name = os.path.basename(file).split("_predictions.csv")[0]
    df = pd.read_csv(file)

    if 'text' not in df.columns:
        raise ValueError(f"'text' column not found in {file}")

    prediction_cols = [col for col in df.columns if col != 'text']
    if not prediction_cols:
        raise ValueError(f"No prediction column found in {file}")
    
    prediction_col = prediction_cols[0]

    df = df[['text', prediction_col]].rename(columns={prediction_col: f'{model_name}_prediction'})
    dfs.append(df)

predictions_df = reduce(lambda left, right: pd.merge(left, right, on='text', how='outer'), dfs)

final_df = pd.merge(test_df, predictions_df, on='text', how='left')

In [ ]:
final_df

In [ ]:
bert_df = pd.read_csv("../results/bert_predictions.csv")

if 'text' not in bert_df.columns or 'true_label' not in bert_df.columns:
    raise ValueError("Missing 'text' or 'true_label' in bert_predictions.csv")

In [ ]:
true_labels_df = bert_df[['text', 'true_label']].drop_duplicates()

In [ ]:
true_labels_df

In [ ]:
final_df = final_df.drop(columns=['my_label'], errors='ignore')
final_df = pd.merge(final_df, true_labels_df, on='text', how='left')

In [ ]:
cols_to_drop = [
    'original_label_description',
    'source',
    'gemma2_prediction',
    'gemma3_prediction',
    'predicted_label_roberta'
]

final_df = final_df.drop(columns=cols_to_drop, errors='ignore')

print(final_df.columns)

In [ ]:
final_df

**Misclassification DeBERTa (high-risk-> neutral)**

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
mismatch_df = final_df[
    (final_df['predicted_label_deberta'] == 2) & (final_df['true_label'] == 0)
]
print(mismatch_df['text'].to_string(index=False))